In [ ]:
%load_ext autoreload
%autoreload 2
import os
if not hasattr(__builtins__, '_cwd_set'):
    os.chdir('..')
    __builtins__._cwd_set = True

In [ ]:
# The issue is not your code. The repository names I gave are incorrect/outdated.

# First, verify your Hugging Face connection:
from huggingface_hub import HfApi

api = HfApi()

# This should print your username (or raise if not authenticated).
print(api.whoami())

In [ ]:

# Then search for the current repository name:
models = api.list_models(search="Qwen3-14B-GGUF", limit=20)

for m in models:
    print(m.id)

In [ ]:
hf download hf://Qwen/Qwen3-14B-GGUF/Qwen3-14B-Q4_K_M.gguf

In [2]:
from pathlib import Path

from huggingface_hub import hf_hub_download
from llama_cpp import Llama

/home/vco/.pyenv/versions/dcrcontroller/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# -------------------------
# Select model
# -------------------------

# MODEL_REPO = "Qwen/Qwen3-14B-GGUF"
# MODEL_FILE = "Qwen3-14B-Q4_K_M.gguf"
MODEL_DIR = Path(f"models/local_llm/")

# Examples:
# MODEL_REPO = "bartowski/Llama-3.3-70B-Instruct-GGUF"
# MODEL_FILE = "Llama-3.3-70B-Instruct-Q4_K_M.gguf"
#
MODEL_REPO = "bartowski/mistralai_Mistral-Small-3.2-24B-Instruct-2506-GGUF"
MODEL_FILE = "Mistral-Small-3.2-24B-Instruct-2506.gguf"
#
# MODEL_REPO = "unsloth/DeepSeek-R1-Distill-Qwen-14B-GGUF"
# MODEL_FILE = "DeepSeek-R1-Distill-Qwen-14B.gguf"
model_path = hf_hub_download(
    repo_id=MODEL_REPO,
    filename=MODEL_FILE,
    local_dir=MODEL_DIR,
    local_dir_use_symlinks=False,  # ignored by newer versions but harmless
)

/home/vco/.pyenv/versions/dcrcontroller/lib/python3.13/site-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


RemoteEntryNotFoundError: 404 Client Error. (Request ID: Root=1-6a773e68-087184533997e33045230b30;e86c38a5-0d5c-4408-98ae-4d0be83089b4)

Entry Not Found for url: https://huggingface.co/bartowski/mistralai_Mistral-Small-3.2-24B-Instruct-2506-GGUF/resolve/main/Mistral-Small-3.2-24B-Instruct-2506.gguf.

In [ ]:
# model_path = "/home/vco/Projects2026/DcrController/backend/models/local_llm/Qwen3-14B-Q4_K_M.gguf"
llm = Llama(
    model_path=model_path,
    n_ctx=8192,
    n_gpu_layers=-1,      # all layers that fit on GPU
    n_threads=16,
    verbose=False,
)

messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant."
    },
    {
        "role": "user",
        "content": "Explain Markov Automata in two paragraphs."
    },
]

response = llm.create_chat_completion(
    messages=messages,
    temperature=0.2,
    max_tokens=512,
)

print(response["choices"][0]["message"]["content"])

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

MODEL = "BAAI/bge-m3"

embedder = SentenceTransformer(
    MODEL,
    device="cuda"   # change to "cpu" if needed
)

documents = [
    "DCR graphs are declarative process models.",
    "Petri nets are a formalism for concurrent systems."
]

vectors = embedder.encode(
    documents,
    normalize_embeddings=True
)

print(vectors.shape)

query = "What are declarative process models?"

q_vector = embedder.encode(
    [query],
    normalize_embeddings=True
)

scores = np.dot(vectors, q_vector.T)

print(scores)

In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb


# Embedding model
embedder = SentenceTransformer(
    "BAAI/bge-m3"
)


# Local vector DB
client = chromadb.PersistentClient(
    path="./chroma_db"
)

collection = client.get_or_create_collection(
    name="research"
)


documents = [
    "DCR graphs are declarative process models.",
    "Petri nets describe concurrent systems.",
    "Markov automata combine CTMC and nondeterminism."
]


embeddings = embedder.encode(
    documents
).tolist()


collection.add(
    ids=["1","2","3"],
    documents=documents,
    embeddings=embeddings
)

query = "How do I model stochastic processes?"

query_embedding = embedder.encode(
    [query]
).tolist()


results = collection.query(
    query_embeddings=query_embedding,
    n_results=2
)


print(results["documents"])

context = "\n".join(
    results["documents"][0]
)

prompt = f"""
Answer using only this context:

{context}

Question:
{query}
"""

In [ ]:
from pathlib import Path

import chromadb

from llama_index.core import (
    VectorStoreIndex,
    StorageContext,
    Settings,
)

from llama_index.core import Document

from llama_index.vector_stores.chroma import ChromaVectorStore

from llama_index.embeddings.huggingface import HuggingFaceEmbedding

from llama_index.llms.llama_cpp import LlamaCPP


# -----------------------------
# Embedding model
# -----------------------------

Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-m3"
)


# -----------------------------
# Local Qwen3 GGUF
# -----------------------------

Settings.llm = LlamaCPP(
    model_path="/models/Qwen3-14B-Q4_K_M.gguf",
    temperature=0.2,
    max_new_tokens=512,
    context_window=8192,
    model_kwargs={
        "n_gpu_layers": -1
    }
)


# -----------------------------
# Chroma database
# -----------------------------

client = chromadb.PersistentClient(
    path="./chroma_db"
)

collection = client.get_or_create_collection(
    "research"
)

vector_store = ChromaVectorStore(
    chroma_collection=collection
)

storage_context = StorageContext.from_defaults(
    vector_store=vector_store
)


# -----------------------------
# Add documents
# -----------------------------

docs = [
    Document(
        text="DCR graphs are declarative process models."
    ),
    Document(
        text="Timed automata represent clocks and constraints."
    ),
    Document(
        text="Markov automata combine CTMC and nondeterminism."
    ),
]


index = VectorStoreIndex.from_documents(
    docs,
    storage_context=storage_context,
)


# -----------------------------
# Query
# -----------------------------

engine = index.as_query_engine(
    similarity_top_k=3
)


response = engine.query(
    "How are stochastic process models represented?"
)

print(response)

In [ ]:
from langchain_community.vectorstores import Chroma

from langchain_community.embeddings import HuggingFaceBgeEmbeddings

from langchain_community.llms import LlamaCpp


# -----------------------------
# Embeddings
# -----------------------------

embedding = HuggingFaceBgeEmbeddings(
    model_name="BAAI/bge-m3"
)


# -----------------------------
# Vector database
# -----------------------------

db = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embedding
)


retriever = db.as_retriever(
    search_kwargs={
        "k": 3
    }
)


# -----------------------------
# Qwen3
# -----------------------------

llm = LlamaCpp(
    model_path="/models/Qwen3-14B-Q4_K_M.gguf",
    n_gpu_layers=-1,
    n_ctx=8192,
)


# -----------------------------
# RAG chain
# -----------------------------

from langchain.chains import RetrievalQA


qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
)


answer = qa.invoke(
    {
        "query":
        "Explain DCR graphs compared to Petri nets."
    }
)


print(answer["result"])